### A descriptor that returns a constant

a descriptor whose `__get__()` method always returns the constant 10:

In [1]:
class Ten:
    def __get__(self, obj, objtype=None):
        return 10

To use the ***descriptor***, it must be stored as a ***class variable*** in another class:

In [2]:
class A:
    x = 5                       # Regular class attribute
    y = Ten()                   # Descriptor instance

In [3]:
a = A()                     # Make an instance of class A

In [4]:
a.x                         # Normal attribute lookup

5

In [5]:
a.y                         # Descriptor lookup

10

In the `a.x` attribute lookup, the dot operator finds `'x': 5` in the class dictionary. In the `a.y` lookup, the dot operator finds a descriptor instance, recognized by its `__get__` method. Calling that method returns 10.

### Dynamic lookups

***Interesting descriptors typically run computations*** instead of returning constants:

In [6]:
import os

class DirectorySize:

    def __get__(self, obj, objtype=None):
        return len(os.listdir(obj.dirname))

class Directory:

    size = DirectorySize()              # Descriptor instance

    def __init__(self, dirname):
        self.dirname = dirname          # Regular instance attribute

In [9]:
s = Directory('songs')
g = Directory('games')

In [13]:
s.size                              # The songs directory has one file

1

In [14]:
g.size                              # The games directory has 0 files

0

In [15]:
os.remove('songs/test.txt')            # Delete a game

In [16]:
g.size                              # File count is automatically updatedf

0

Besides showing how descriptors can run computations, this example also reveals the purpose of the parameters to `__get__()`. The `self` parameter is `size`, an instance of `DirectorySize`. The `obj` parameter is either `g` or `s`, an instance of `Directory`. It is the `obj` parameter that lets the `__get__()` method learn the target directory. The `objtype` parameter is the class `Directory`.

### Managed attributes

***A popular use for descriptors is managing access to instance data***. The descriptor is assigned to a public attribute in the class dictionary while the actual data is stored as a private attribute in the instance dictionary. The descriptor’s `__get__()` and `__set__()` methods are triggered when the public attribute is accessed.

`age` is the public attribute and `_age` is the private attribute. When the public attribute is accessed, the descriptor logs the lookup or update:

In [17]:
import logging

logging.basicConfig(level=logging.INFO)

class LoggedAgeAccess:

    def __get__(self, obj, objtype=None):
        value = obj._age
        logging.info('Accessing %r giving %r', 'age', value)
        return value

    def __set__(self, obj, value):
        logging.info('Updating %r to %r', 'age', value)
        obj._age = value

class Person:

    age = LoggedAgeAccess()             # Descriptor instance

    def __init__(self, name, age):
        self.name = name                # Regular instance attribute
        self.age = age                  # Calls __set__()

    def birthday(self):
        self.age += 1                   # Calls both __get__() and __set__()

In [18]:
mary = Person('Mary M', 30)         # The initial age update is logged

INFO:root:Updating 'age' to 30


In [19]:
dave = Person('David D', 40)

INFO:root:Updating 'age' to 40


In [20]:
vars(mary)                          # The actual data is in a private attribute

{'name': 'Mary M', '_age': 30}

In [21]:
vars(dave)

{'name': 'David D', '_age': 40}

In [22]:
mary.age                            # Access the data and log the lookup

INFO:root:Accessing 'age' giving 30


30

In [23]:
mary.birthday()                     # Updates are logged as well

INFO:root:Accessing 'age' giving 30
INFO:root:Updating 'age' to 31


In [24]:
dave.name                           # Regular attribute lookup isn't logged

'David D'

In [25]:
dave.age                            # Only the managed attribute is logged

INFO:root:Accessing 'age' giving 40


40

One major issue with this example is that the private name `_age` is ***hardwired*** in the `LoggedAgeAccess` class. That means that each instance can only have one logged attribute and that its name is ***unchangeable***. In the next example, we’ll fix that problem.

### Customized names

***When a class uses descriptors, it can inform each descriptor about which variable name was used.***

In [26]:
import logging

logging.basicConfig(level=logging.INFO)

class LoggedAccess:

    def __set_name__(self, owner, name):
        self.public_name = name
        self.private_name = '_' + name

    def __get__(self, obj, objtype=None):
        value = getattr(obj, self.private_name)
        logging.info('Accessing %r giving %r', self.public_name, value)
        return value

    def __set__(self, obj, value):
        logging.info('Updating %r to %r', self.public_name, value)
        setattr(obj, self.private_name, value)

class Person:

    name = LoggedAccess()                # First descriptor instance
    age = LoggedAccess()                 # Second descriptor instance

    def __init__(self, name, age):
        self.name = name                 # Calls the first descriptor
        self.age = age                   # Calls the second descriptor

    def birthday(self):
        self.age += 1

`Person` class has two descriptor instances, `name` and `age`. When the `Person` class is defined, it makes a callback to `__set_name__()` in `LoggedAccess` so that the ***field names can be recorded***, giving each descriptor its ***own public_name and private_name***:

In [30]:
Person.__dict__

mappingproxy({'__module__': '__main__',
              'name': <__main__.LoggedAccess at 0x1997e094e50>,
              'age': <__main__.LoggedAccess at 0x1997e3575d0>,
              '__init__': <function __main__.Person.__init__(self, name, age)>,
              'birthday': <function __main__.Person.birthday(self)>,
              '__dict__': <attribute '__dict__' of 'Person' objects>,
              '__weakref__': <attribute '__weakref__' of 'Person' objects>,
              '__doc__': None})

In [31]:
vars(Person.__dict__['name'])

{'public_name': 'name', 'private_name': '_name'}

In [28]:
vars(vars(Person)['age'])

{'public_name': 'age', 'private_name': '_age'}

`Person` class has called `__set_name__()` so that the field names would be recorded. Here we call `vars()` to look up the descriptor without triggering it

In [36]:
pete = Person('Peter P', 10)

INFO:root:Updating 'name' to 'Peter P'
INFO:root:Updating 'age' to 10


The new class now logs access to both `name` and `age`

In [33]:
kate = Person('Catherine C', 20)

INFO:root:Updating 'name' to 'Catherine C'
INFO:root:Updating 'age' to 20


The two Person instances contain only the private names:

In [35]:
print(vars(pete))
print(vars(kate))

{'_name': 'Peter P', '_age': 10}
{'_name': 'Catherine C', '_age': 20}


### Closing thoughts

A descriptor is what we call any object that defines `__get__()`, `__set__()`, or `__delete__()`.

Optionally, descriptors can have a `__set_name__()` method. This is only used in cases where a descriptor needs to know either the class where it was created or the name of class variable it was assigned to. (This method, if present, is called even if the class is not a descriptor.)

***Descriptors get invoked by the dot operator during attribute lookup***. If a descriptor is accessed indirectly with vars(some_class)[descriptor_name], the descriptor instance is returned without invoking it.

***Descriptors only work when used as class variables***. When put in instances, they have no effect.

The main motivation for descriptors is to provide a hook allowing objects stored in class variables to control what happens during attribute lookup.

Traditionally, the calling class controls what happens during lookup. Descriptors invert that relationship and allow the data being looked-up to have a say in the matter.

Descriptors are used throughout the language. It is how functions turn into bound methods. Common tools like `classmethod()`, `staticmethod()`, `property()`, and `functools.cached_property()` are all implemented as descriptors.

### Complete Practical Example

A practical and powerful tool for locating notoriously hard to find data corruption bugs.

#### Validator class

A validator is a descriptor for managed attribute access. Prior to storing any data, it verifies that the new value meets various type and range restrictions. If those restrictions aren’t met, it raises an exception to prevent data corruption at its source.

This `Validator` class is both an ***abstract base class*** and a managed attribute ***descriptor***:

In [39]:
from abc import ABC, abstractmethod

class Validator(ABC):

    def __set_name__(self, owner, name):
        self.private_name = '_' + name

    def __get__(self, obj, objtype=None):
        return getattr(obj, self.private_name)

    def __set__(self, obj, value):
        self.validate(value)
        setattr(obj, self.private_name, value)

    @abstractmethod
    def validate(self, value):
        pass

Custom validators need to inherit from `Validator` and must supply a `validate()` method to test various restrictions as needed.

#### Custom validators

Here are three practical data validation utilities:

* 1. `OneOf` verifies that a `value` is one of a restricted set of options.

* 2. `Number` verifies that a `value` is either an `int` or `float`. Optionally, it verifies that a value is between a given minimum or maximum.

* 3. `String` verifies that a `value` is a `str`. Optionally, it validates a given minimum or maximum length. It can validate a user-defined predicate as well.

In [40]:
class OneOf(Validator):

    def __init__(self, *options):
        self.options = set(options)

    def validate(self, value):
        if value not in self.options:
            raise ValueError(f'Expected {value!r} to be one of {self.options!r}')

In [42]:
class Number(Validator):

    def __init__(self, minvalue=None, maxvalue=None):
        self.minvalue = minvalue
        self.maxvalue = maxvalue

    def validate(self, value):
        if not isinstance(value, (int, float)):
            raise TypeError(f'Expected {value!r} to be an int or float')
        if self.minvalue is not None and value < self.minvalue:
            raise ValueError(
                f'Expected {value!r} to be at least {self.minvalue!r}'
            )
        if self.maxvalue is not None and value > self.maxvalue:
            raise ValueError(
                f'Expected {value!r} to be no more than {self.maxvalue!r}'
            )

In [43]:
class String(Validator):

    def __init__(self, minsize=None, maxsize=None, predicate=None):
        self.minsize = minsize
        self.maxsize = maxsize
        self.predicate = predicate

    def validate(self, value):
        if not isinstance(value, str):
            raise TypeError(f'Expected {value!r} to be an str')
        if self.minsize is not None and len(value) < self.minsize:
            raise ValueError(
                f'Expected {value!r} to be no smaller than {self.minsize!r}'
            )
        if self.maxsize is not None and len(value) > self.maxsize:
            raise ValueError(
                f'Expected {value!r} to be no bigger than {self.maxsize!r}'
            )
        if self.predicate is not None and not self.predicate(value):
            raise ValueError(
                f'Expected {self.predicate} to be true for {value!r}'
            )

#### Practical application

Here’s how the data validators can be used in a real class:

In [44]:
class Component:

    name = String(minsize=3, maxsize=10, predicate=str.isupper)
    kind = OneOf('wood', 'metal', 'plastic')
    quantity = Number(minvalue=0)

    def __init__(self, name, kind, quantity):
        self.name = name
        self.kind = kind
        self.quantity = quantity

The descriptors prevent invalid instances from being created:

In [45]:
Component('Widget', 'metal', 5)      # Blocked: 'Widget' is not all uppercase

ValueError: Expected <method 'isupper' of 'str' objects> to be true for 'Widget'

In [46]:
Component('WIDGET', 'metle', 5)      # Blocked: 'metle' is misspelled

ValueError: Expected 'metle' to be one of {'wood', 'plastic', 'metal'}

In [47]:
Component('WIDGET', 'metal', -5)     # Blocked: -5 is negative

ValueError: Expected -5 to be at least 0

In [48]:
Component('WIDGET', 'metal', 'V')    # Blocked: 'V' isn't a number

TypeError: Expected 'V' to be an int or float

In [49]:
c = Component('WIDGET', 'metal', 5)  # Allowed:  The inputs are valid